# Day 5 · Part 2 — Loading and QC of real airway RNA-seq counts

Today we work with the **real bulk RNA-seq count matrix** from Himes et al. 2014 (GSE52778 / SRP033351).

**Source:** Recount2 (https://jhubiostatistics.shinyapps.io/recount/) — reanalyzed to hg38 with a uniform pipeline.

**Pipeline today:**
1. Download the count matrix and phenotype data
2. Build a clean sample metadata table
3. Library size + gene detection QC
4. Filter low-count genes
5. Normalize (CPM, log2)
6. PCA + sample-sample correlation
7. Save processed data for Day 6

## 1. Setup

Run this cell once. It installs `mygene` (for ENSEMBL → symbol mapping).

> ⚠️ **Heads up:** Colab may pop up a *Restart session* dialog after this install. Click **Restart**, then re-run the cells from the top. This is normal Colab behavior.

In [ ]:
!pip install mygene --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request
import gzip
import os

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
print('Setup complete.')

## 2. Download the real count matrix from Recount3

Recount3 (the newer version of Recount2) hosts all 16 airway runs as a gene-level count matrix. ~2 MB gzipped TSV.

Genes are rows (GENCODE ENSEMBL IDs), samples are columns (SRR run IDs).

In [ ]:
COUNTS_URL = 'https://recount-opendata.s3.amazonaws.com/recount3/release/human/data_sources/sra/gene_sums/51/SRP033351/sra.gene_sums.SRP033351.G026.gz'

if not os.path.exists('airway_recount3.tsv.gz'):
    urllib.request.urlretrieve(COUNTS_URL, 'airway_recount3.tsv.gz')
print('Counts file downloaded.')
!ls -lh airway_recount3.tsv.gz

In [ ]:
# Load the count matrix (skip Recount3 header comment lines that start with #)
counts_raw = pd.read_csv('airway_recount3.tsv.gz', sep='\t', comment='#', index_col=0)
print('Shape (genes x samples):', counts_raw.shape)
print('Samples present:', list(counts_raw.columns))
counts_raw.head()

All **16 runs** are present (the full Untreated / Dex / Albuterol / Albu+Dex design).

**Important detail about Recount3:** the values you just loaded are *base-pair coverage*, not read counts. To convert to approximate read counts (which is what DESeq2 wants), we divide by the read length. The airway study used **126 bp paired-end reads** (you can verify this in the SRA Run Selector under the `AvgSpotLen` column).

In [ ]:
READ_LENGTH = 126   # from SRA Run Selector AvgSpotLen
counts = (counts_raw / READ_LENGTH).round().astype(int)
print(f'{counts.shape[0]:,} genes x {counts.shape[1]} samples')
print(f'Library sizes range: {counts.sum().min()/1e6:.1f}M - {counts.sum().max()/1e6:.1f}M reads')

## 3. Build the sample metadata and subset to 6 samples

We hardcode the 6 Untreated-vs-Dex SRR runs from the SRA Run Selector you browsed yesterday — keeping 3 donor pairs from the classic untreated-vs-dex comparison and dropping the 8 Albuterol-containing runs and 1 donor pair.

(In a real project you would parse this from `SraRunTable.txt`, but here it is small enough to write by hand.)

In [ ]:
metadata = pd.DataFrame({
    'sample':    ['SRR1039512', 'SRR1039513',
                  'SRR1039516', 'SRR1039517',
                  'SRR1039520', 'SRR1039521'],
    'cell_line': ['N052611', 'N052611',
                  'N080611', 'N080611',
                  'N061011', 'N061011'],
    'condition': ['control', 'dex',
                  'control', 'dex',
                  'control', 'dex'],
}).set_index('sample')
metadata

Subset the count matrix to just these 6 samples and check alignment:

In [ ]:
counts = counts[metadata.index]
assert (counts.columns == metadata.index).all(), 'Sample order mismatch!'
print('Subset counts:', counts.shape, '   Metadata:', metadata.shape)
print(f'Dropped {counts_raw.shape[1] - counts.shape[1]} samples; kept 6 Untreated/Dex samples (3 donor pairs).')

## 4. Library size QC

Total reads per sample. Big differences mean we have to normalize before comparing samples.

In [ ]:
lib_sizes = counts.sum(axis=0) / 1e6   # millions of reads

fig, ax = plt.subplots(figsize=(8, 4))
colors = metadata['condition'].map({'control': '#028090', 'dex': '#F45B69'})
ax.bar(range(len(lib_sizes)), lib_sizes, color=colors)
ax.set_xticks(range(len(lib_sizes)))
ax.set_xticklabels(lib_sizes.index, rotation=45, ha='right')
ax.set_ylabel('Library size (millions of reads)')
ax.set_title('Library size per sample')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#028090', label='control'), Patch(color='#F45B69', label='dex')])
plt.tight_layout(); plt.show()

### 🔬 Try it yourself — Exercise 1

Use pandas to answer:
1. Which sample has the **largest** library size?
2. Which sample has the **smallest**?
3. What is the ratio between the largest and smallest? (If it's >2, we definitely need to normalize before comparing.)

In [ ]:
# Your code here
# Hint: lib_sizes.idxmax(), lib_sizes.idxmin(), lib_sizes.max() / lib_sizes.min()



## 5. Gene detection QC

How many genes are detected (count > 0) per sample?

In [ ]:
genes_detected = (counts > 0).sum(axis=0)
genes_detected.to_frame('genes_detected')

## 6. Filter low-count genes

A gene is only useful if it has reasonable expression in at least a few samples. Standard rule: keep genes with **≥10 counts in ≥3 samples** (half the cohort).

In [ ]:
keep = (counts >= 10).sum(axis=1) >= 3
counts_filt = counts.loc[keep]
print(f'Kept {keep.sum():,} of {len(keep):,} genes ({100*keep.mean():.1f}%)')

## 7. Normalize: CPM and log2(CPM+1)

**CPM** = Counts Per Million reads — corrects for library size differences. We add `+1` before log so zeros don't blow up.

In [ ]:
cpm = counts_filt.div(counts_filt.sum(axis=0), axis=1) * 1e6
log_cpm = np.log2(cpm + 1)
log_cpm.head()

### 🔬 Try it yourself — Exercise 2

Pick **any one gene** from `counts_filt` (e.g., the first one). For one sample (e.g., `SRR1039516`):
1. Print its raw count
2. Print its CPM value
3. Verify by hand: `CPM = raw_count / library_size × 1,000,000`. Do the numbers match?

In [ ]:
# Your code here
# gene = counts_filt.index[0]
# sample = 'SRR1039516'
# raw = counts_filt.loc[gene, sample]
# cpm_value = cpm.loc[gene, sample]
# manual = raw / counts_filt[sample].sum() * 1e6
# print(f'raw={raw}, cpm={cpm_value:.2f}, manual={manual:.2f}')



Distribution of log2(CPM+1) per sample — should look similar across samples after normalization:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
log_cpm.boxplot(ax=ax, rot=45)
ax.set_ylabel('log2(CPM + 1)')
ax.set_title('Expression distribution per sample (after CPM normalization)')
plt.tight_layout(); plt.show()

## 8. Sample-sample correlation

Samples from the same condition should correlate more strongly with each other. We use the top variable genes for a cleaner signal.

In [ ]:
# Pick the 1000 most variable genes
gene_var = log_cpm.var(axis=1)
top_var = gene_var.sort_values(ascending=False).head(1000).index

corr = log_cpm.loc[top_var].corr(method='pearson')

row_colors = metadata['condition'].map({'control': '#028090', 'dex': '#F45B69'})
g = sns.clustermap(corr, cmap='RdBu_r', vmin=0.95, vmax=1.0, center=0.975,
                   row_colors=row_colors, col_colors=row_colors,
                   figsize=(7, 6), cbar_kws={'label': 'Pearson r'})
g.fig.suptitle('Sample-sample correlation (top 1000 variable genes)', y=1.02)
plt.show()

## 9. PCA on samples

If the treatment effect is real, PC1 or PC2 should separate control vs dex.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = log_cpm.loc[top_var].T.values   # samples x genes
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=4)
pcs = pca.fit_transform(X_scaled)
var_exp = pca.explained_variance_ratio_ * 100
print('Variance explained:', [f'{v:.1f}%' for v in var_exp])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for cond, color in [('control', '#028090'), ('dex', '#F45B69')]:
    mask = (metadata['condition'] == cond).values
    ax.scatter(pcs[mask, 0], pcs[mask, 1], color=color, s=120, label=cond,
               edgecolor='black', linewidth=1.2)
for i, sample in enumerate(metadata.index):
    ax.annotate(metadata.loc[sample, 'cell_line'], (pcs[i, 0], pcs[i, 1]),
                xytext=(6, 6), textcoords='offset points', fontsize=9)
ax.set_xlabel(f'PC1 ({var_exp[0]:.1f}%)')
ax.set_ylabel(f'PC2 ({var_exp[1]:.1f}%)')
ax.set_title('PCA — airway samples')
ax.legend()
plt.tight_layout(); plt.show()

**Q1.** Which PC separates the conditions? Is it PC1 or PC2?

**Q2.** Does PC1 (or PC2) tell you anything about the donor (cell line)?

### 🔬 Try it yourself — Exercise 3

1. How much **total variance** do PC1 + PC2 explain together? (Sum the first two values of `var_exp`.)
2. Re-make the same PCA scatter but color by **`cell_line`** instead of `condition`. Is there donor-specific clustering on any PC?
3. *(Bonus)* Make a scatter of **PC3 vs PC4**. Do the conditions separate there too?

In [ ]:
# Your code here



## 10. Map ENSEMBL IDs to gene symbols

Recount2 uses ENSEMBL gene IDs. For interpretation we want gene symbols (e.g., `FKBP5`, `DUSP1`). We use the `mygene` package to query biothings.io.

*(Recount IDs may have a `.NN` version suffix like `ENSG00000096060.13` — strip it before querying.)*

In [ ]:
import mygene
mg = mygene.MyGeneInfo()

# Strip version suffix
ensembl_clean = log_cpm.index.str.split('.').str[0]

# Query in batches
result = mg.querymany(ensembl_clean.tolist(), scopes='ensembl.gene',
                       fields='symbol', species='human', returnall=False, verbose=False)

id_to_symbol = {r['query']: r.get('symbol', None) for r in result if not r.get('notfound')}
print(f'Mapped {len(id_to_symbol):,} of {len(ensembl_clean):,} IDs to gene symbols')

In [ ]:
# Add a 'symbol' column to log_cpm for convenience
log_cpm_named = log_cpm.copy()
log_cpm_named.index = ensembl_clean
log_cpm_named['symbol'] = log_cpm_named.index.map(id_to_symbol)
log_cpm_named[['symbol'] + list(metadata.index)].head()

## 11. Spot-check known glucocorticoid response genes

If our data is good, **FKBP5** (a classic dex-induced gene) should be higher in dex samples.

In [ ]:
def plot_gene(symbol):
    rows = log_cpm_named[log_cpm_named['symbol'] == symbol]
    if rows.empty:
        print(f'{symbol}: not found')
        return
    expr = rows.iloc[0][metadata.index]
    df = metadata.copy()
    df['expr'] = expr.values
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.stripplot(data=df, x='condition', y='expr', hue='condition',
                  palette={'control': '#028090', 'dex': '#F45B69'},
                  size=12, ax=ax, legend=False)
    ax.set_title(f'{symbol} expression')
    ax.set_ylabel('log2(CPM + 1)')
    plt.tight_layout(); plt.show()

plot_gene('FKBP5')
plot_gene('DUSP1')
plot_gene('KLF15')

**Q3.** Do FKBP5, DUSP1, and KLF15 show higher expression in dex-treated samples? (They should — these are textbook glucocorticoid-induced genes.)

### 🔬 Try it yourself — Exercise 4

**Sanity check with housekeeping genes.** A *housekeeping gene* is one that should be expressed at roughly constant levels regardless of treatment — examples: **`ACTB`** (β-actin), **`GAPDH`**, **`HPRT1`**.

1. Use `plot_gene()` to plot **ACTB**, **GAPDH**, and **HPRT1**.
2. Are they roughly equal between control and dex? *(They should be — if they're not, our normalization may be off.)*

In [ ]:
# Your code here
# plot_gene('ACTB')
# plot_gene('GAPDH')
# plot_gene('HPRT1')



## 12. Save processed data for Day 6

In [ ]:
counts_filt.to_csv('airway_counts_filtered.csv')
metadata.to_csv('airway_metadata.csv')
pd.Series(id_to_symbol).to_csv('ensembl_to_symbol.csv', header=['symbol'])
print('Saved: airway_counts_filtered.csv, airway_metadata.csv, ensembl_to_symbol.csv')
!ls -lh airway_*.csv ensembl_*.csv

---
## Recap

You just ran a **complete bulk RNA-seq QC + normalization pipeline on real published data** — the same pipeline used in published papers.

| Step | What you did |
|---|---|
| Download | Pulled real counts from Recount2 |
| QC | Library sizes, gene detection, sample-sample correlation |
| Filter | Removed low-count genes |
| Normalize | CPM + log2 transformation |
| Visualize | PCA showed clear treatment separation |
| Annotate | ENSEMBL → symbol via mygene |

**Tomorrow:** PyDESeq2 → significant genes → volcano plot → heatmap → pathway enrichment.